# 大規模エージェント: Agent Runtime 上の A2A プロトコルと ADK 統合によるマルチエージェント アーキテクチャ

## 概要

AI エージェントが担う責任が増えるにつれ、単一のエージェントですべてを処理することは保守、スケーリング、進化が困難になります。異なる機能には、異なるデプロイ戦略、更新サイクル、あるいは異なるチームによる所有が必要になることがよくあります。

* [Agent2Agent（A2A）プロトコル](https://a2a-protocol.org/latest/) は通信面を解決します — エージェントが互いの機能を発見し、フレームワークや組織を超えて連携する方法を標準化します。
* [Gemini Enterprise Agent Platform Runtime](https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/overview) はデプロイ面を解決します — 組み込みの A2A サポート、自動スケーリング、セキュアなエンドポイント、永続セッション、インフラストラクチャ管理不要を備え、エージェントをホストするフルマネージドのサーバーレス プラットフォームです。

これらを組み合わせることで、特化したエージェントを構築し、検出可能な A2A サービスとしてデプロイし、マルチエージェント システムに統合することができます。

## 構築するもの

Google Cloud Run にデプロイされた在庫 MCP サーバー（[`mcp_tool_agent.ipynb`](mcp_tool_agent.ipynb) で作成した `inventory-mcp-server`）に接続する**在庫エージェント（Inventory Agent）**。このエージェントを A2A サービスとして Gemini Enterprise Agent Platform Runtime にデプロイします。次に、デプロイされた在庫エージェントをリモート A2A サブエージェントとして利用する**店舗管理者エージェント（Store Manager Agent: `store_manager_agent`）**を構築します。その結果、店舗管理者エージェントが一般的な店舗への問い合わせを処理し、在庫の問い合わせや在庫更新をリモートの A2A 在庫エージェントにルーティングする、エンタープライズ マルチエージェント アーキテクチャが実現します。

## 学習目標

* `mcp_tool_agent.ipynb` で構築した MCP サーバーの Cloud Run デプロイを確認する。
* リモート Cloud Run MCP サーバーに接続された `McpToolset` を使用して ADK エージェントを構築する。
* MCP を利用したエージェントを、エージェント カードとスキルを備えた A2A サーバーとして公開する。
* A2A エージェントを Gemini Enterprise Agent Runtime にデプロイする。
* `RemoteA2aAgent` を使用して、別の ADK エージェントからリモート A2A エージェントを利用する。
* ローカル A2A と ADK Web UI を使用してマルチエージェント システムを段階的にテストする。

> [!IMPORTANT]
> **前提条件:** このノートブックは [`mcp_tool_agent.ipynb`](mcp_tool_agent.ipynb) でデプロイされた Cloud Run MCP サーバーに依存しています。このノートブックを実行する前に、必ず `mcp_tool_agent.ipynb` を実行して `inventory-mcp-server` サービスをデプロイしてください。


## 環境のセットアップ

必要なパッケージをインポートし、Google Cloud プロジェクトとロケーションの環境変数を設定します。


In [ ]:
import asyncio
import logging
import os
import sys
import uuid
import warnings
import subprocess
import importlib

import google.auth
import vertexai

from a2a.helpers import new_text_message
from a2a.server.context import ServerCallContext
from a2a.types import (
    AgentCard,
    AgentSkill,
    GetExtendedAgentCardRequest,
    GetTaskRequest,
    Message,
    Part,
    Role,
    SendMessageRequest,
    TaskState,
)
from google.adk.a2a.utils.agent_to_a2a import (
    A2aAgentExecutor,
)
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.auth import default
from google.genai import types
from vertexai.agent_engines.templates.a2a import A2aAgent, create_agent_card

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.ERROR)
logging.getLogger("opentelemetry.context").setLevel(logging.CRITICAL)

In [ ]:
# Google Cloud プロジェクトとロケーションの設定
_, PROJECT_ID = google.auth.default()
LOCATION = "global"
REGION = "us-central1"
CLOUD_RUN_LOCATION = "us-central1"
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["REGION"] = REGION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"  # Agent Platform API を使用


### デプロイ済み Cloud Run MCP サーバーの確認

A2A エージェントを構築する前に、[`mcp_tool_agent.ipynb`](mcp_tool_agent.ipynb) でデプロイされた `inventory-mcp-server` Cloud Run サービスがアクティブであることを確認し、そのサービス URL を取得します。サービスが見つからない場合はエラーが発生します。


In [ ]:
cmd = [
    "gcloud", "run", "services", "describe", "inventory-mcp-server",
    "--region", CLOUD_RUN_LOCATION,
    "--project", PROJECT_ID,
    "--format", "value(status.url)"
]

try:
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    cloud_run_url = result.stdout.strip()
    if not cloud_run_url:
        raise RuntimeError("Cloud Run service URL returned empty.")
    MCP_SERVER_URL = f"{cloud_run_url}/sse"
    os.environ["REMOTE_MCP_SERVER_URL"] = MCP_SERVER_URL
    print(f"Verified Cloud Run MCP Server URL: {MCP_SERVER_URL}")
except Exception as e:
    raise RuntimeError(
        "Could not find deployed 'inventory-mcp-server' Cloud Run service. "
        "Please execute 'mcp_tool_agent.ipynb' first to deploy the Cloud Run MCP server."
    ) from e

## 概念: Agent2Agent（A2A）プロトコルと Gemini Enterprise Agent Runtime

### Agent2Agent（A2A）プロトコル
[Agent2Agent（A2A）プロトコル](https://a2a-protocol.org) は、個別に構築された AI エージェント間の通信と相互運用性を実現するために設計されたオープン標準です。独自の API や厳格なコードレベルのインポートを介してエージェントを接続するのではなく、A2A はエージェントの検出、インタラクション、タスク委託のための統一されたインターフェースを定義します。

エージェントをツールとしてラップする（MCP 経由）場合と A2A 経由で公開する場合の主な違い: ツールはステートレスで単一の機能を実行するのに対し、A2A エージェントは推論を行い、状態を維持し、交渉や確認などのマルチターンの対話を処理できます。A2A 経由で公開されたエージェントは、関数呼び出しに縮小されることなく、その完全な機能を保持します。

A2A は 3 つのコア概念を定義しています:
1. **Agent Card（エージェント カード）** — エージェントの機能、スキル、エンドポイントを記述した JSON ドキュメント。他のエージェントはこのカードを取得して機能を発見します。
2. **Message（メッセージ）** — A2A エンドポイントに送信されるユーザーまたはエージェントのリクエストで、タスクをトリガーします。
3. **Task（タスク）** — ライフサイクル（submitted → working → completed/failed）と結果を含む **artifacts（アーティファクト）** を持つ作業単位。

### Gemini Enterprise Agent Platform Runtime
**Agent Runtime** は、エンタープライズ セキュリティ機能（VPC Service Controls、CMEK など）を備え、本番環境で AI エージェントをデプロイ、スケーリング、管理するための Google Cloud 上のフルマネージド サービスです。インフラストラクチャを処理するため、エージェントのロジックに集中できます。

Agent Runtime が提供するもの:
* **マネージド デプロイ** — ADK、LangGraph、または任意の Python フレームワークで構築されたエージェントを、単一の SDK 呼び出しでデプロイ
* **A2A ホスティング** — 自動エージェント カード提供と認証済みアクセスを備えた A2A 準拠エンドポイントとしてエージェントをデプロイ
* **永続セッション** — `VertexAiSessionService` がリクエスト間で会話履歴と状態を保存
* **自動スケーリング** — インフラストラクチャ管理不要で、トラフィックに合わせてゼロから自動スケーリング
* **オブザーバビリティ** — Google Cloud のオブザーバビリティ スタックを介した組み込みのトレース、ロギング、モニタリング

このチュートリアルでは、在庫エージェントを Agent Runtime にデプロイします。デプロイ プロセスではエージェント コードがシリアライズ（pickle 化）されてアップロードされます。Agent Runtime は A2A プロトコルを提供するサーバーレス エンドポイントをプロビジョニングし、他のエージェント（またはクライアント）は Google Cloud 認証情報で認証された標準の HTTP 呼び出しを介してやり取りします。


## A2A プロトコルによる在庫エージェントの公開


### 既存の在庫 MCP エージェント（`custom_mcp_agent.agent.agent`）のインポート

`mcp_tool_agent.ipynb` で作成した `custom_mcp_agent/agent/agent.py` から `root_agent` を直接インポートします。このエージェントは、Cloud Run MCP サーバーに接続された `McpToolset` を使用します。


In [ ]:
if "." not in sys.path:
    sys.path.insert(0, ".")

import custom_mcp_agent.agent.agent
importlib.reload(custom_mcp_agent.agent.agent)
from custom_mcp_agent.agent.agent import root_agent

print(f"Loaded ADK Agent: {root_agent.name}")
print(f"Description: {root_agent.description}")
print(f"MCP Server URL: {custom_mcp_agent.agent.agent.MCP_SERVER_URL}")

### クラウド デプロイ用の A2A エージェントの準備（`A2aAgent` + `A2aAgentExecutor`）とローカルでのテスト

**Google Cloud Agent Runtime**（`client.agent_engines.create(...)`）は、`HTTP+JSON` インターフェース バインディングを持つ `vertexai.agent_engines.templates.a2a.A2aAgent` テンプレート クラスを前提としています。

カスタム エグゼキュータをゼロから作成する代わりに、ADK の組み込み **`A2aAgentExecutor`**（`from google.adk.a2a.utils.agent_to_a2a import A2aAgentExecutor`）を `A2aAgent` のエグゼキュータ ビルダーとして使用します。

このステップでは、`A2aAgent` + `A2aAgentExecutor` を使用して在庫エージェントをラップし、ノートブック内でローカルに完全な A2A プロトコルのライフサイクルをテストします:
1. **エージェント カードの取得**（`on_get_extended_agent_card`） — 機能とスキルの検出。
2. **メッセージの送信**（`on_message_send`） — タスクを作成するリクエストの送信。
3. **タスクのステータスとアーティファクト**（`on_get_task`） — タスクの完了をポーリングし、結果を抽出。
4. **セッションの継続性** — リクエスト間で `contextId` を再利用し、エージェントが前回のやり取りを記憶できるようにする。


#### モック A2A リクエスト用ヘルパー関数の定義

標準的な A2A デプロイでは、Agent Runtime サーバーは受信した HTTP リクエストを Starlette の `Request` オブジェクトに変換します。外部 Web サーバーを起動せずにローカルで `A2aAgent` をテストするために、受信 HTTP リクエストをシミュレートし、タスクのステータスをポーリングするヘルパー関数（`build_get_request`、`build_post_request`、`wait_for_task`）を定義します。


In [ ]:
# SDK の protobuf 型を使用してローカルで A2A プロトコル メソッドをテストするためのヘルパー関数
def create_call_context() -> ServerCallContext:
    """ローカルでのメソッド呼び出し用にデフォルトの ServerCallContext を作成します。"""
    return ServerCallContext()


def build_send_message_request(text: str, context_id: str = "") -> SendMessageRequest:
    """ユーザーのテキスト メッセージを含む SendMessageRequest を構築します。"""
    return SendMessageRequest(
        message=Message(
            message_id=f"msg-{uuid.uuid4().hex[:8]}",
            role=Role.ROLE_USER,
            parts=[Part(text=text)],
            context_id=context_id,
        )
    )


async def wait_for_task(a2a_agent, task_id: str, context: ServerCallContext, max_retries: int = 30):
    """タスクが完了（completed）または失敗（failed）状態になるまで on_get_task をポーリングします。"""
    for _ in range(max_retries):
        result = await a2a_agent.on_get_task(
            request=GetTaskRequest(id=task_id),
            context=context,
        )
        if result and result.status.state in [
            TaskState.TASK_STATE_COMPLETED,
            TaskState.TASK_STATE_FAILED,
        ]:
            return result
        await asyncio.sleep(1)
    return result


def print_task_answer(result):
    """タスク アーティファクトからテキスト レスポンスを抽出して出力します。"""
    if not result:
        print("No task result returned.")
        return
    print(f"Status: {TaskState.Name(result.status.state)}")
    for artifact in result.artifacts:
        for part in artifact.parts:
            if part.text:
                print(f"Answer: {part.text}")


#### ローカル A2A エージェントの初期化

`agent_card` と ADK の組み込み `A2aAgentExecutor` を渡して `A2aAgent` をインスタンス化し、`set_up()` を呼び出して ADK ランナーとインメモリ セッション サービスを初期化します。


In [ ]:
import sys
import importlib

from custom_mcp_agent.agent.agent import root_agent

# 1. デプロイ用にエージェント テンプレートが pickle 化可能な状態を維持できるよう、ADK Runner の関数ビルダーを準備
def get_inventory_runner():
    import importlib
    import custom_mcp_agent.agent.agent
    importlib.reload(custom_mcp_agent.agent.agent)
    from custom_mcp_agent.agent.agent import root_agent
    return Runner(
        app_name=root_agent.name,
        agent=root_agent,
        session_service=InMemorySessionService(),
    )

# 2. Google Cloud Agent Runtime 用の AgentCard を構築（HTTP+JSON バインディングが必要）
skills = [
    AgentSkill(
        id="inventory_management",
        name="inventory_management",
        description="Query inventory items (list SKUs) and update stock quantities.",
        tags=["inventory", "mcp"],
        input_modes=["text/plain"],
        output_modes=["text/plain"],
    )
]
agent_card = create_agent_card(
    agent_name=root_agent.name,
    description=root_agent.description,
    skills=skills,
)

# 3. ADK の組み込み A2aAgentExecutor を使用して A2aAgent を初期化
a2a_agent = A2aAgent(
    agent_card=agent_card,
    agent_executor_builder=A2aAgentExecutor,
    agent_executor_kwargs={"runner": get_inventory_runner},
    extended_agent_card=agent_card,
)
a2a_agent.set_up()
print("=== A2A Agent Successfully Initialized ===")
print(f"Name: {a2a_agent.agent_card.name}")
print(f"Skills: {[skill.name for skill in a2a_agent.agent_card.skills]}")


#### テスト 1: エージェント カードの検出

A2A エコシステム内の他のエージェントは、エージェント カードのエンドポイント（`/v1/card`）に問い合わせることで、エージェントの名前、説明、スキルを検出します。`on_get_extended_agent_card` の呼び出しをテストできます:


In [ ]:
# 1. エージェント カード取得のテスト
card_response = await a2a_agent.on_get_extended_agent_card(
    request=GetExtendedAgentCardRequest(),
    context=create_call_context(),
)

print("=== Agent Card ===")
print(f"Name: {card_response.name}")
print(f"Description: {card_response.description}")
print(f"Skills: {[s.name for s in card_response.skills]}")


#### テスト 2: 在庫 SKU の照会

A2A プロトコルでは、エージェントにメッセージを送信すると、会話タスクが作成または継続されます。在庫内の利用可能な SKU を一覧表示するリクエストを送信してみましょう。


In [ ]:
# 2. A2A メッセージによる SKU 一覧表示のテスト
request = build_send_message_request(
    "List all available SKUs in the inventory."
)
context = create_call_context()
response = await a2a_agent.on_message_send(request=request, context=context)

task_id = response.id
context_id = response.context_id
print(f"Task submitted! Task ID: {task_id}")
print(f"Session Context ID: {context_id}")

# タスクが完了するまでポーリング
result = await wait_for_task(a2a_agent, task_id, context)
print("\n=== Inventory Query Response ===")
print_task_answer(result)


#### テスト 3: 在庫数量の更新（セッションの継続性）

A2A 上でのマルチターン会話とセッションの継続性をテストするには、前のリクエストから返された `context_id` を渡します。SKU の在庫数量を更新してみましょう。


In [ ]:
# 3. セッション継続性のために contextId を使用して SKU 数量を更新するテスト
request = build_send_message_request(
    "Update quantity for SKU001 to 50 items",
    context_id=context_id,
)
update_response = await a2a_agent.on_message_send(
    request=request, context=context
)

update_result = await wait_for_task(a2a_agent, update_response.id, context)
print("=== Update Inventory Response ===")
print_task_answer(update_result)


#### テスト 4: 在庫更新の検証

最後に、更新された数量が A2A プロトコル経由の後続のクエリに反映されていることを確認します。


In [ ]:
# 4. A2A メッセージによる在庫更新確認のテスト
request = build_send_message_request(
    "List all SKUs again to confirm the updated quantity for SKU001.",
    context_id=context_id,
)
verify_response = await a2a_agent.on_message_send(
    request=request, context=context
)

verify_result = await wait_for_task(a2a_agent, verify_response.id, context)
print("=== Verify Inventory Response ===")
print_task_answer(verify_result)


### 在庫エージェントの Agent Runtime へのデプロイ

このステップでは、在庫エージェントを Gemini Enterprise Agent Platform Runtime にデプロイします。Agent Runtime はエージェントをコンテナにパッケージ化し、サーバーレス エンドポイントにデプロイして、A2A プロトコルのルーティングを処理します。


#### Agent Runtime でのエージェントのデプロイと更新

Vertex AI Python SDK を使用して、`A2aAgent` を Gemini Enterprise Agent Platform Runtime にデプロイします。このプロセスは 2 つの段階で構成されます:

1. **初期デプロイ（`client.agent_engines.create`）**:
   Vertex AI は、エージェントのコード、依存関係、MCP ツールセットをコンテナにパッケージ化し、サーバーレス Agent Engine にデプロイします。完了すると、Vertex AI は自動生成されたリソース名（`projects/<project>/locations/<region>/reasoningEngines/<id>`）を割り当てます。

2. **エンドポイントの解決と更新（`client.agent_engines.update`）**:

   **デプロイされたエージェントを更新する理由**
   A2A プロトコルでは、エージェントは `AgentCard`（具体的には `supported_interfaces`）を通じて互いの機能と接続エンドポイントを検出して通信します。Vertex AI Agent Runtime の場合、標準のエンドポイント URL は `https://{REGION}-aiplatform.googleapis.com/v1beta1/{RESOURCE_NAME}/a2a` です。

   しかし、初期デプロイの時点では、コンテナ ランタイムにまだ一意のリソース ID が割り当てられていないため、`AgentCard.supported_interfaces[0].url` はデフォルトでプレースホルダー（`test-agent-engine`）になっています。

   デプロイされたエージェントが完全な自己無撞着性を持ち、実際のエンドポイントをクライアントに公開できるようにするため:
   * エージェントの `supported_interfaces` に実際のエンドポイント URL をバインドします。
   * `client.agent_engines.update()` を呼び出して、デプロイされたエージェントを更新します。

更新されると、デプロイされたサービスは検証済みの独自の `AgentCard` をネイティブに提供します。ダウンストリームのコンシューマー エージェント（店舗管理者など）は、手動で `AgentCard` を定義したりクライアント側で URL を修正したりすることなく、デプロイされたエージェントから直接カードを取得できます。

> **注:** Google Cloud リソースの作成と更新には、各ステップで 3〜5 分程度かかります。


In [ ]:
RESOURCE_NAME = os.environ.get("INVENTORY_AGENT_RESOURCE_NAME", "")

print("Starting deployment to Agent Runtime (this takes 3-5 minutes)...")
STAGING_BUCKET = os.environ.get(
    "STAGING_BUCKET", f"{PROJECT_ID}-adk-a2a-agent-runtime"
)
BUCKET_URI = f"gs://{STAGING_BUCKET}"

CONFIG = {
        "display_name": agent_card.name,
        "description": agent_card.description,
        "requirements": [
            "google-cloud-aiplatform[agent_engines,adk]==1.160.0",
            "google-genai==2.11.0",
            "google-adk[mcp, a2a]==2.5.0",
            "mcp>=1.25.0,<2.0.0",
            "a2a-sdk==1.1.2",
            "cloudpickle",
            "pydantic",
        ],
        "extra_packages": [
            "./custom_mcp_agent",
        ],
        "http_options": {
            "api_version": "v1beta1",
        },
        "staging_bucket": BUCKET_URI,
        "env_vars": {"REMOTE_MCP_SERVER_URL": MCP_SERVER_URL},
    }

vertexai.init(
    project=PROJECT_ID, location=REGION, staging_bucket=BUCKET_URI
)
client = vertexai.Client(
    project=PROJECT_ID,
    location=REGION,
    http_options=types.HttpOptions(api_version="v1beta1"),
)

remote_agent = client.agent_engines.create(
    agent=a2a_agent,
    config=CONFIG
)

RESOURCE_NAME = remote_agent.api_resource.name
os.environ["INVENTORY_AGENT_RESOURCE_NAME"] = RESOURCE_NAME
print(f"\nDeployment complete! Resource name: {RESOURCE_NAME}")

# デプロイされた Agent Engine を更新して、実際のランタイム エンドポイント URL を指すように設定
endpoint_url = f"https://{REGION}-aiplatform.googleapis.com/v1beta1/{RESOURCE_NAME}/a2a"

# a2a_agent のカード インターフェースを実際のデプロイ先 URL で更新
a2a_agent.agent_card.supported_interfaces[0].url = endpoint_url

print(
    f'\nUpdating deployed Agent Engine with real endpoint {endpoint_url} '
    '(this takes 3-5 minutes)...'
)
remote_agent = client.agent_engines.update(
    name=RESOURCE_NAME,
    agent=a2a_agent,
    config=CONFIG,
)
print('Update complete! The deployed AgentCard now natively contains the resolved endpoint URL.')


#### デプロイされたリモート A2A エージェントのテスト

デプロイ後は、認可された任意のクライアントが HTTP 経由でエージェントとやり取りできます。リモート エージェント カードを取得し、`client.agent_engines.get(...)` を使用して在庫リクエストを送信する例を示します:


In [ ]:
# 1. リモート エージェント カードの取得
card = await remote_agent.on_get_extended_agent_card(
    request=GetExtendedAgentCardRequest()
)
print(f"Connected to Remote Agent: {card.name}")

# 2. リモート エージェントへのメッセージ送信
print("\nSending inventory request...")
req = SendMessageRequest(
    message=new_text_message(
        "List all SKUs available in inventory."
    )
)
response = await remote_agent.on_message_send(request=req)

# 3. タスク レスポンスの出力
for item in response:
    task = item.task
    if hasattr(task, "artifacts") and task.artifacts:
        for art in task.artifacts:
            for part in art.parts:
                text = getattr(part, "text", None) or (
                    part.root.text if hasattr(part, "root") else None
                )
                if text:
                    print(f"\nResponse:\n{text}")


## A2A エージェントの利用


### A2A 在庫エージェントと店舗管理者エージェントの統合

このステップでは、ADK の `RemoteA2aAgent` を使用して在庫エージェントをリモート A2A サブエージェントとして利用する**店舗管理者エージェント（Store Manager Agent: `store_manager_agent`）**を作成します。

統合すると:
* 一般的な店舗への問い合わせ（"What are the store hours?"（営業時間は何時ですか？））は、ローカルの店舗ツールによって処理されます。
* 在庫の問い合わせや更新（"Check stock of Wireless Headphones"（ワイヤレスヘッドホンの在庫を確認して）や "Update SKU001 quantity to 50"（SKU001 の数量を 50 に更新して））は、A2A プロトコル経由で `RemoteA2aAgent` に自動的に委託されます。


#### デプロイされたエージェントからのエージェント カードの取得

Agent Engine が実際のエンドポイント URL で更新されたので、`remote_agent.on_get_extended_agent_card()` を介してデプロイされたエージェントから直接 `AgentCard` を取得します。

`card.supported_interfaces[0].url` が、デプロイされた実際の Agent Runtime エンドポイントを直接指していることに注目してください。この URL を `INVENTORY_AGENT_CARD_URL` としてエクスポートし、店舗管理者コンシューマー エージェントがネットワーク経由で動的にカードを取得できるようにします:


In [ ]:
# 1. デプロイされたエージェントから直接リモート エージェント カードを取得
card = await remote_agent.on_get_extended_agent_card(
    request=GetExtendedAgentCardRequest()
)
print(f"Connected to Remote Agent: {card.name}")
print(f"Deployed Agent Endpoint URL: {card.supported_interfaces[0].url}")

# コンシューマー エージェントがデプロイされたエージェントの extendedAgentCard エンドポイントを直接指すように設定
INVENTORY_AGENT_CARD_URL = f"{card.supported_interfaces[0].url}/extendedAgentCard"
os.environ["INVENTORY_AGENT_CARD_URL"] = INVENTORY_AGENT_CARD_URL


#### 店舗管理者エージェントの作成（`store_manager_agent/agent.py`）

`./a2a_agent/store_manager_agent/agent.py` を作成して、ルート コンシューマー エージェント（`store_manager_agent`）を定義します。このファイルでは以下を設定します:

1. **ローカル店舗ツール（`get_store_info`）**: 営業時間、連絡先情報、所在地などの一般的な店舗への問い合わせを処理するローカル Python 関数。
2. **認証済み HTTP クライアント（`get_gcp_httpx_client`）**: Google アプリケーションのデフォルト認証情報（ADC: `google.auth.default()`）を使用して OAuth ベアラ トークンを生成し、リモート A2A Agent Runtime サービスとの安全な通信を確保します。
3. **リモート A2A サブエージェント（`RemoteA2aAgent`）**: デプロイされたエージェントのエンドポイント（`INVENTORY_AGENT_CARD_URL`）から直接、リモート A2A `inventory_assistant` の機能を動的に解決します。
4. **オーケストレーター エージェント（`root_agent`）**: `get_store_info` と `AgentTool(inventory_remote_agent)` を備えた、Gemini 3.5 Flash を利用した `LlmAgent`。ユーザーが在庫について質問すると、店舗管理者は A2A プロトコル経由で AgentTool を使用してリモートの `inventory_assistant` にクエリを自動的に委託します。


In [ ]:
# コンシューマー エージェント ファイル用のディレクトリ構造を作成
!mkdir -p ./a2a_agent/store_manager_agent


In [ ]:
%%writefile ./a2a_agent/store_manager_agent/__init__.py
# store_manager_agent 用のパッケージ初期化ファイル


In [ ]:
%%writefile ./a2a_agent/store_manager_agent/agent.py
# store_manager_agent/agent.py
"""A2A 統合用の店舗管理者エージェント モジュール。"""

import os

import httpx
from google.adk.agents import LlmAgent
from google.adk.agents.remote_a2a_agent import RemoteA2aAgent
from google.adk.models.google_llm import Gemini
from google.adk.tools import AgentTool
from google.auth import default
from google.auth.transport.requests import Request as AuthRequest

INVENTORY_AGENT_CARD_URL = os.environ.get("INVENTORY_AGENT_CARD_URL")
if not INVENTORY_AGENT_CARD_URL:
    raise ValueError(
        "INVENTORY_AGENT_CARD_URL environment variable is not set. "
        "It must point to the deployed Agent Engine endpoint."
    )


def get_store_info(info_type: str = "hours") -> str:
    """営業時間や連絡先詳細などの一般的な店舗情報を取得します。"""
    if "hour" in str(info_type).lower():
        return (
            "Store hours: Monday-Saturday 8:00 AM - 9:00 PM, "
            "Sunday 10:00 AM - 6:00 PM."
        )
    if "contact" in str(info_type).lower():
        return (
            "Store contact: support@store.example.com, Phone: (555) 019-2834."
        )
    return "Store location: 123 Main Street, Suite 100."


def get_gcp_httpx_client(timeout: int = 60) -> httpx.AsyncClient:
    """ADC を使用して認証済み HTTP クライアントを作成します。"""
    scopes = ["https://www.googleapis.com/auth/cloud-platform"]
    credentials, _ = default(scopes=scopes)
    if not credentials.valid:
        credentials.refresh(AuthRequest())
    return httpx.AsyncClient(
        headers={"Authorization": f"Bearer {credentials.token}"},
        timeout=timeout,
    )


# リモート A2A エージェントはデプロイされた Agent Engine エンドポイントから AgentCard を動的に取得します
inventory_remote_agent = RemoteA2aAgent(
    name="inventory_assistant",
    agent_card=INVENTORY_AGENT_CARD_URL,
    httpx_client=get_gcp_httpx_client(),
)


root_agent = LlmAgent(
    name="store_manager_agent",
    model=Gemini(
        model="gemini-3.5-flash",
        client_kwargs={"vertexai": True, "location": "global"},
    ),
    instruction=(
        "You are a Store Manager Assistant responsible for overall store "
        "operations. Your job:\n"
        "- Provide store information such as opening hours, contact details, "
        "and store policies using store tools.\n"
        "- For any inventory questions, listing SKUs, checking item stock, "
        "or updating item quantities, delegate to the inventory_assistant.\n\n"
        "Be conversational, helpful, and concise."
    ),
    tools=[get_store_info, AgentTool(inventory_remote_agent)],
)


In [ ]:
%%writefile ./a2a_agent/store_manager_agent/.env
# Gemini モデルと Vertex AI ADC のグローバル ロケーションを指定
GOOGLE_CLOUD_LOCATION=global
GOOGLE_GENAI_USE_VERTEXAI=TRUE


#### 統合エージェントのテスト
ADK Developer UI（`adk web`）を使用して、統合された店舗管理者エージェントをテストできます。


In [ ]:
# Cloud Workstations の場合
!adk web a2a_agent/store_manager_agent --allow_origins "regex:https://.*\.cloudworkstations\.dev"


**注**: Agent Platform Workbench を使用している場合は、コメントアウトを解除して以下のセルを実行してください。


In [ ]:
# %%bash
# PROXY_BASE=$(curl -s http://metadata.google.internal/computeMetadata/v1/instance/attributes/proxy-url -H "Metadata-Flavor: Google")
# echo "--------------------------------------------------------"
# echo "🔗 ACCESS HERE: https://${PROXY_BASE}/proxy/8000/dev-ui/"
# echo "--------------------------------------------------------"
# adk web a2a_agent/store_manager_agent --url_prefix /proxy/8000  --allow_origins "regex:https://.*\.notebooks\.googleusercontent\.com"


Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at

https://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.